In [2]:
import os
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
import getpass
load_dotenv(".env.RAG")

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embd = GoogleGenerativeAIEmbeddings(model="models/gemini-2.5-flash")






True

In [ ]:
# ! pip install -U langchain-huggingface

In [3]:

# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2907.55it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# file_path = "./Data/Res.pdf"
folder_path="./Data"

loader = DirectoryLoader(
    folder_path,
    glob="**/*",


)


docs = loader.load()
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Hunter Jacobson Human Resources Manager Human Resources student looking to leverage my experience in recruiting for technical roles to help a mission-driven company like Panorama grow their team when 

{'source': 'Data\\res10.pdf'}


- used clear_func to formate the text for better embeding

In [11]:

import re
def cleaner_func(docs):

  if isinstance(docs, list):
    cleared=[]
    for doc in docs:
      if isinstance(doc, str):
         doc = re.sub(r'\n+', '\n', doc)
         doc = re.sub(r'\s+', ' ', doc)
         cleared.append(doc.strip())
      else:
        cleared.append(doc)
    return cleared

    

  elif isinstance(docs, str):
    doc= ''.join(str(d)for d in docs)

  docs = re.sub(r'\n+', '\n', docs)
  docs = re.sub(r'\s+', ' ', docs)
  return docs.strip()

cleaned_docs = cleaner_func(docs)

In [ ]:
from langchain_chroma import Chroma



In [15]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":7}
)



In [ ]:
# ! pip install -U huggingface_hub

In [7]:
hf_token=os.getenv("HF_TOKEN")

In [ ]:
# ! pip install rank_bm25

In [13]:
from rank_bm25 import BM25Okapi

In [8]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3000.54it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [25]:
pipline= pipeline(
    "text-generation",
    model="microsoft/phi-2",
    max_new_tokens=512,
    temperature=0.3
)

Loading weights: 100%|██████████| 453/453 [00:00<00:00, 2506.16it/s]
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
llm = HuggingFacePipeline(pipeline=pipline) 
template = """Given the following context, answer the question accurately.
Context: {context}
Question: {question}
Answer:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])
qa_chain = LLMChain(llm=llm, prompt=prompt)

C:\Users\ghali\AppData\Local\Temp\ipykernel_13260\728503515.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=prompt)


In [28]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,  
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
    length_function=len,
)
cleaned_chunk = text_split.split_documents(cleaned_docs)


vector_store = Chroma.from_documents(
    documents=cleaned_chunk,
    collection_name="Lang_Rag_hf",
    embedding=embd,
    persist_directory="./Rag/chroma_DB",
    
)

text_chunks = [chunk.page_content for chunk in cleaned_chunk]
tokenized_chunks = [chunk.split() for chunk in text_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def retrieve(query, top_k=5):
    vector_docs = retriever.invoke(query)
    vector_texts = [doc.page_content for doc in vector_docs]
    
    tokenized_query = query.split()
    bm25_texts = bm25.get_top_n(tokenized_query, text_chunks, n=top_k)
    
    all_texts = []
    seen = set()
    for text in bm25_texts + vector_texts:
        if text not in seen:
            seen.add(text)
            all_texts.append(text)
    
    return all_texts[:top_k*2]

def rerank(query, texts):
    if 'reranker' not in globals():
        return texts[:3]
    
    pairs = [[query, text] for text in texts]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, texts), reverse=True)
    return [text for score, text in ranked[:3]]

def ask(question):
    
    texts = retrieve(question)
    
    top_texts = rerank(question, texts)
    
    if 'qa_chain' in globals():
        context = "\n\n".join(top_texts)
        response = qa_chain.run(context=context, question=question)
        if "Answer:" in response:
            response = response.split("Answer:")[-1].strip()
        return response
    else:
        return f"Found {len(top_texts)} relevant passages."

print(ask("Which university does Hunter Jacobson studies mentioned in the resume? He is a student of human resource"))

C:\Users\ghali\AppData\Local\Temp\ipykernel_13260\1078941903.py:57: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain.run(context=context, question=question)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hunter Jacobson is a student of human resource management at the University of Colorado.
